> title : 건설공사 사고 예방 및 대응책 생성 : 한솔데코 시즌3 AI 경진대회 <br>
> author : hjy <br>

- https://dacon.io/competitions/official/236455/overview/description
- https://github.com/dmskorea/project7-Hansol-Deco-Season-3-AI-Competition

<img src="https://dacon.s3.ap-northeast-2.amazonaws.com/competition/236455/header_background.jpeg" style="width:100%; height:auto;">


In [1]:
!pip uninstall -y scipy
!pip install -q scipy==1.13.0
!pip install -q -U gensim --no-deps
!pip install -q -U bitsandbytes
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U git+https://github.com/huggingface/peft.git
!pip install -q -U git+https://github.com/huggingface/accelerate.git
!pip install -q -U datasets ipywidgets

Found existing installation: scipy 1.13.1
Uninstalling scipy-1.13.1:
  Successfully uninstalled scipy-1.13.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━

In [2]:
import pandas as pd
import numpy as np
import os
import sys
import re
from sklearn.model_selection import train_test_split

import torch
import transformers
from datetime import datetime
from transformers import AutoModelForCausalLM, AutoTokenizer,BitsAndBytesConfig
from datasets import load_dataset
from peft import prepare_model_for_kbit_training,LoraConfig,PeftModel,get_peft_model

from accelerate import FullyShardedDataParallelPlugin, Accelerator
from torch.distributed.fsdp.fully_sharded_data_parallel import FullOptimStateDictConfig, FullStateDictConfig

fsdp_plugin = FullyShardedDataParallelPlugin(
    state_dict_config=FullStateDictConfig(offload_to_cpu=True, rank0_only=False),
    optim_state_dict_config=FullOptimStateDictConfig(offload_to_cpu=True, rank0_only=False),
)
accelerator = Accelerator(fsdp_plugin=fsdp_plugin)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
pd.set_option("display.max_columns", None)  # 모든 컬럼 출력
pd.set_option("display.max_colwidth", 20)  # 컬럼 내용 생략 없음
pd.set_option('display.max_rows', 100)

#### 📂 read data

In [5]:
# path = '/kaggle/input/project7'
path = '/content/drive/MyDrive'

# [1]
# train = pd.read_csv(f'{path}/train_new.csv', encoding = 'utf-8-sig')
# valid = pd.read_csv(f'{path}/valid.csv', encoding = 'utf-8-sig')
# test = pd.read_csv(f'{path}/test_new.csv', encoding = 'utf-8-sig')
# submission = pd.read_csv(f'{path}/sample_submission.csv', encoding = 'utf-8-sig')

# [2]
data = pd.read_csv('/content/drive/MyDrive/train.csv', encoding = 'utf-8-sig')
test = pd.read_csv('/content/drive/MyDrive/test.csv', encoding = 'utf-8-sig')

# 컬럼명 변경
data = data.rename(columns={'사고인지 시간':'사고인지시간'})
test = test.rename(columns={'사고인지 시간':'사고인지시간'})
data = data.rename(columns={'재발방지대책 및 향후조치계획':'재발방지대책'})
test['재발방지대책'] = None

#### 📂 데이터 전처리

In [6]:
# 발생일자, 발생월
data['발생일자'] = data['발생일시'].str[:10]
data['발생월'] = data['발생일시'].str[:7]
test['발생일자'] = test['발생일시'].str[:10]
test['발생월'] = test['발생일시'].str[:7]

In [7]:
# 사고원인 유형별 재분류
# -> #부주의 #넘어짐 #미끄러짐 #작업미숙

data['사고원인유형_알수없음'] = np.where(data['사고원인'].isin(['.','-','미상','원인미상','작업자','알수없음']),'알수없음','')
data['사고원인유형_알수없음'] = np.where(data['사고원인'].str.contains('알수없음'),'알수없음',data['사고원인유형_알수없음'])
data['사고원인유형_조사중'] = np.where(data['사고원인'].str.contains('조사|파악 중'),'조사중','')
data['사고원인유형_기타'] = np.where(data['사고원인'].str.contains('기타|해당없음'),'기타','')
data['사고원인유형_작업자부주의'] = np.where(data['사고원인'].str.contains('부주의|부주위'),'작업자부주의','')
data['사고원인유형_과실'] = np.where(data['사고원인'].str.contains('과실|실수'),'작업자부주의','')
data['사고원인유형_불안전한행동'] = np.where(data['사고원인'].str.contains('불안전한 행동|불안전한 작업자세|불안전한|불완전한|무리한|무리하게|부적절한|부적정'),'작업자부주의','')
data['사고원인유형_넘어짐'] = np.where(data['사고원인'].str.contains('넘어짐|발걸림'),'넘어짐','')
data['사고원인유형_미끄러짐'] = np.where(data['사고원인'].str.contains('미끄러'),'미끄러짐','')
data['사고원인유형_추락낙상'] = np.where(data['사고원인'].str.contains('추락|낙상'),'추락','')
data['사고원인유형_발헛디딤'] = np.where(data['사고원인'].str.contains('헛디딤|헛딛음|디뎌'),'발헛디딤','')
data['사고원인유형_작업미숙'] = np.where(data['사고원인'].str.contains('작업미숙|미숙|미흡'),'작업미숙','')
data['사고원인유형_작업방법불량'] = np.where(data['사고원인'].str.contains('작업방법 불량'),'작업방법불량','')
data['사고원인유형_근골격계질환'] = np.where(data['사고원인'].str.contains('근골격계'),'근골격계','')
data['사고원인유형_질병'] = np.where(data['사고원인'].str.contains('질병|심장마비'),'질병','')
data['사고원인유형_안전불량'] = np.where(data['사고원인'].str.contains('안전|안정|미착용|미사용|미실시|미준수|미확보|미확인|미 실시|미 확인|미설치|비규격화|전방주의 부족|확인 부족'),'안전불량','')
data['사고원인유형_오작동'] = np.where(data['사고원인'].str.contains('오작동'),'오작동','')
data['사고원인유형_작업중이동'] = np.where(data['사고원인'].str.contains('이동'),'작업중이동','')
data['사고원인유형_작업태도불량'] = np.where(data['사고원인'].str.contains('불량|주의태만|관리소홀'),'작업태도불량','')
data['사고원인유형_신호불일치'] = np.where(data['사고원인'].str.contains('불일치'),'신호불일치','')
data['사고원인유형_정리정돈'] = np.where(data['사고원인'].str.contains('정리'),'정리','')
data['사고원인유형_타격'] = np.where(data['사고원인'].str.contains('타격|타박'),'타격','')
data['사고원인유형_끼임'] = np.where(data['사고원인'].str.contains('끼임'),'끼임','')
data['사고원인유형_부딪힘'] = np.where(data['사고원인'].str.contains('부딪힘'),'부딪힘','')
data['사고원인유형_그라인더'] = np.where(data['사고원인'].str.contains('그라인더|그라인드'),'그라인더','')
data['사고원인유형_화재'] = np.where(data['사고원인'].str.contains('화재'),'화재','')
data['사고원인유형_바닥'] = np.where(data['사고원인'].str.contains('바닥'),'바닥','')
data['사고원인유형_중심잃음'] = np.where(data['사고원인'].str.contains('중심을 잃음'),'중심잃음','')
data['사고원인유형_교통사고'] = np.where(data['사고원인'].str.contains('교통'),'교통','')
data['사고원인유형_협착'] = np.where(data['사고원인'].str.contains('협착'),'협착','')
data['사고원인유형_골절'] = np.where(data['사고원인'].str.contains('골절'),'골절','')
data['사고원인유형_절단'] = np.where(data['사고원인'].str.contains('절단'),'절단','')
data['사고원인유형_손가락'] = np.where(data['사고원인'].str.contains('손가락'),'손가락','')
data['사고원인유형_발목'] = np.where(data['사고원인'].str.contains('발목'),'발목','')
data['사고원인유형_발등'] = np.where(data['사고원인'].str.contains('발등'),'발등','')
data['사고원인유형_허리'] = np.where(data['사고원인'].str.contains('허리'),'허리','')
data['사고원인유형_피로'] = np.where(data['사고원인'].str.contains('피로|쓰러짐|무리'),'피로','')

feats = [i for i in data.columns if '사고원인유형_' in i]
data['사고원인유형'] = data[feats].apply(lambda x:' '.join(sorted(x)).strip().replace(' ','|'),axis=1)
data['사고원인유형'] = np.where(data['사고원인유형']=='','미분류',data['사고원인유형'])
data = data.drop(columns=feats)
print('# 미분류(data):',np.round((data['사고원인유형']=='미분류').sum()/len(data)*100,2),'%')

test['사고원인유형_알수없음'] = np.where(test['사고원인'].isin(['.','-','미상','원인미상','작업자','알수없음']),'알수없음','')
test['사고원인유형_알수없음'] = np.where(test['사고원인'].str.contains('알수없음'),'알수없음',test['사고원인유형_알수없음'])
test['사고원인유형_조사중'] = np.where(test['사고원인'].str.contains('조사|파악 중'),'조사중','')
test['사고원인유형_기타'] = np.where(test['사고원인'].str.contains('기타|해당없음'),'기타','')
test['사고원인유형_작업자부주의'] = np.where(test['사고원인'].str.contains('부주의|부주위'),'작업자부주의','')
test['사고원인유형_과실'] = np.where(test['사고원인'].str.contains('과실|실수'),'작업자부주의','')
test['사고원인유형_불안전한행동'] = np.where(test['사고원인'].str.contains('불안전한 행동|불안전한 작업자세|불안전한|불완전한|무리한|무리하게|부적절한|부적정'),'작업자부주의','')
test['사고원인유형_넘어짐'] = np.where(test['사고원인'].str.contains('넘어짐|발걸림'),'넘어짐','')
test['사고원인유형_미끄러짐'] = np.where(test['사고원인'].str.contains('미끄러'),'미끄러짐','')
test['사고원인유형_추락낙상'] = np.where(test['사고원인'].str.contains('추락|낙상'),'추락','')
test['사고원인유형_발헛디딤'] = np.where(test['사고원인'].str.contains('헛디딤|헛딛음|디뎌'),'발헛디딤','')
test['사고원인유형_작업미숙'] = np.where(test['사고원인'].str.contains('작업미숙|미숙|미흡'),'작업미숙','')
test['사고원인유형_작업방법불량'] = np.where(test['사고원인'].str.contains('작업방법 불량'),'작업방법불량','')
test['사고원인유형_근골격계질환'] = np.where(test['사고원인'].str.contains('근골격계'),'근골격계','')
test['사고원인유형_질병'] = np.where(test['사고원인'].str.contains('질병|심장마비'),'질병','')
test['사고원인유형_안전불량'] = np.where(test['사고원인'].str.contains('안전|안정|미착용|미사용|미실시|미준수|미확보|미확인|미 실시|미 확인|미설치|비규격화|전방주의 부족|확인 부족'),'안전불량','')
test['사고원인유형_오작동'] = np.where(test['사고원인'].str.contains('오작동'),'오작동','')
test['사고원인유형_작업중이동'] = np.where(test['사고원인'].str.contains('이동'),'작업중이동','')
test['사고원인유형_작업태도불량'] = np.where(test['사고원인'].str.contains('불량|주의태만|관리소홀'),'작업태도불량','')
test['사고원인유형_신호불일치'] = np.where(test['사고원인'].str.contains('불일치'),'신호불일치','')
test['사고원인유형_정리정돈'] = np.where(test['사고원인'].str.contains('정리'),'정리','')
test['사고원인유형_타격'] = np.where(test['사고원인'].str.contains('타격|타박'),'타격','')
test['사고원인유형_끼임'] = np.where(test['사고원인'].str.contains('끼임'),'끼임','')
test['사고원인유형_부딪힘'] = np.where(test['사고원인'].str.contains('부딪힘'),'부딪힘','')
test['사고원인유형_그라인더'] = np.where(test['사고원인'].str.contains('그라인더|그라인드'),'그라인더','')
test['사고원인유형_화재'] = np.where(test['사고원인'].str.contains('화재'),'화재','')
test['사고원인유형_바닥'] = np.where(test['사고원인'].str.contains('바닥'),'바닥','')
test['사고원인유형_중심잃음'] = np.where(test['사고원인'].str.contains('중심을 잃음'),'중심잃음','')
test['사고원인유형_교통사고'] = np.where(test['사고원인'].str.contains('교통'),'교통','')
test['사고원인유형_협착'] = np.where(test['사고원인'].str.contains('협착'),'협착','')
test['사고원인유형_골절'] = np.where(test['사고원인'].str.contains('골절'),'골절','')
test['사고원인유형_절단'] = np.where(test['사고원인'].str.contains('절단'),'절단','')
test['사고원인유형_손가락'] = np.where(test['사고원인'].str.contains('손가락'),'손가락','')
test['사고원인유형_발목'] = np.where(test['사고원인'].str.contains('발목'),'발목','')
test['사고원인유형_발등'] = np.where(test['사고원인'].str.contains('발등'),'발등','')
test['사고원인유형_허리'] = np.where(test['사고원인'].str.contains('허리'),'허리','')
test['사고원인유형_피로'] = np.where(test['사고원인'].str.contains('피로|쓰러짐|무리'),'피로','')

feats = [i for i in test.columns if '사고원인유형_' in i]
test['사고원인유형'] = test[feats].apply(lambda x:' '.join(sorted(x)).strip().replace(' ','|'),axis=1)
test['사고원인유형'] = np.where(test['사고원인유형']=='','미분류',test['사고원인유형'])
test = test.drop(columns=feats)
print('# 미분류(test):',np.round((test['사고원인유형']=='미분류').sum()/len(test)*100,2),'%')

# 미분류(data): 18.44 %
# 미분류(test): 18.05 %


In [8]:
# 공사종류, 공종, 사고객체 -> 대분류, 중분류로 파생
data['공사종류(대분류)'] = data['공사종류'].str.split(' / ').str[0]
data['공사종류(중분류)'] = data['공사종류'].str.split(' / ').str[1]
data['공종(대분류)'] = data['공종'].str.split(' > ').str[0]
data['공종(중분류)'] = data['공종'].str.split(' > ').str[1]
data['사고객체(대분류)'] = data['사고객체'].str.split(' > ').str[0]
data['사고객체(중분류)'] = data['사고객체'].str.split(' > ').str[1]
test['공사종류(대분류)'] = test['공사종류'].str.split(' / ').str[0]
test['공사종류(중분류)'] = test['공사종류'].str.split(' / ').str[1]
test['공종(대분류)'] = test['공종'].str.split(' > ').str[0]
test['공종(중분류)'] = test['공종'].str.split(' > ').str[1]
test['사고객체(대분류)'] = test['사고객체'].str.split(' > ').str[0]
test['사고객체(중분류)'] = test['사고객체'].str.split(' > ').str[1]

# 장소
data['장소(대분류)'] = data['장소'].str.split('/').str[0].str.strip()
data['장소(중분류)'] = data['장소'].str.split('/').str[1].str.strip()
test['장소(대분류)'] = test['장소'].str.split('/').str[0].str.strip()
test['장소(중분류)'] = test['장소'].str.split('/').str[1].str.strip()

# 부위
data['부위(대분류)'] = data['부위'].str.split('/').str[0].str.strip()
data['부위(중분류)'] = data['부위'].str.split('/').str[1].str.strip()
test['부위(대분류)'] = test['부위'].str.split('/').str[0].str.strip()
test['부위(중분류)'] = test['부위'].str.split('/').str[1].str.strip()

# 발생일시
def preprocess_datetime(s):
    import datetime
    d, m, t = s.split()
    dt = d+" "+t
    dt = datetime.datetime.strptime(dt, "%Y-%m-%d %H:%M")
    if m == "오후" and dt.hour != 12:
        dt = dt + datetime.timedelta(hours = 12)
    return dt

data['발생일시'] = data['발생일시'].map(preprocess_datetime)
test['발생일시'] = test['발생일시'].map(preprocess_datetime)

data['사고발생시간'] = data['발생일시'].dt.hour
test['사고발생시간'] = test['발생일시'].dt.hour

weekday_map = {0: '월', 1: '화', 2: '수', 3: '목', 4: '금', 5: '토', 6: '일'}
data['사고발생요일'] = data['발생일시'].dt.day_of_week.map(weekday_map)
test['사고발생요일'] = test['발생일시'].dt.day_of_week.map(weekday_map)

data['사고발생월'] = data['발생일시'].dt.month
test['사고발생월'] = test['발생일시'].dt.month

In [9]:
# 6:4 비율로 데이터 분할
train, valid = train_test_split(data, test_size=0.4, random_state=42)

# 결과 출력
print(f"# train: {len(train)}")
print(f"# valid: {len(valid)}")
print(f"# test: {len(test)}")

# train: 14053
# valid: 9369
# test: 964


#### 📂 question-answer 전처리

In [10]:
combined_train_data = train.apply(
    lambda row: {
        "question": (
            f"공사종류 대분류 '{row['공사종류(대분류)']}', 중분류 '{row['공사종류(중분류)']}' 공사 중 "
            f"공종 대분류 '{row['공종(대분류)']}', 중분류 '{row['공종(중분류)']}' 작업에서 "
            f"사고객체 '{row['사고객체(대분류)']}'(중분류: '{row['사고객체(중분류)']}')와 관련된 사고가 발생했습니다. "
            f"작업 프로세스는 '{row['작업프로세스']}'이며, 사고 원인은 '{row['사고원인']}'이고 사고 원인 유형은 '{row['사고원인유형']}' 유형 입니다. "
            f"조사결과 본 사고의 인적사고유형은 '{row['인적사고']}'이며, 물적사고유형은 '{row['물적사고']}'입니다. "
            f"장소 대분류 '{row['장소(대분류)']}', 장소 중분류 '{row['장소(중분류)']}'에서 "
            f"부위 대분류 '{row['부위(대분류)']}', 부위 중분류 '{row['부위(중분류)']}' 사고가 발생하였습니다. "
            f"해당 사고 발생 당시 기온은 '{row['기온']}'이며, 발생일시는 '{row['발생일시']}'입니다. "
            f"사고발생시간은 {row['사고발생시간']}시 이며, 사고요일은 {row['사고발생요일']}요일 이고, 사고발생월은 {row['사고발생월']}월 입니다. "
            f"재발 방지 대책 및 향후 조치 계획은 무엇인가요?"
        ),
        "context": (
            f"공사종류 대분류 '{row['공사종류(대분류)']}', 중분류 '{row['공사종류(중분류)']}' 공사 중 "
            f"공종 대분류 '{row['공종(대분류)']}', 중분류 '{row['공종(중분류)']}' 작업에서 "
            f"사고객체 '{row['사고객체(대분류)']}'(중분류: '{row['사고객체(중분류)']}')와 관련된 사고가 발생했습니다. "
            f"작업 프로세스는 '{row['작업프로세스']}'이며, 사고 원인은 '{row['사고원인']}'이고 사고 원인 유형은 '{row['사고원인유형']}' 유형 입니다. "
            f"조사결과 본 사고의 인적사고유형은 '{row['인적사고']}'이며, 물적사고유형은 '{row['물적사고']}'입니다. "
            f"장소 대분류 '{row['장소(대분류)']}', 장소 중분류 '{row['장소(중분류)']}'에서 "
            f"부위 대분류 '{row['부위(대분류)']}', 부위 중분류 '{row['부위(중분류)']}' 사고가 발생하였습니다. "
            f"해당 사고 발생 당시 기온은 '{row['기온']}'이며, 발생일시는 '{row['발생일시']}'입니다. "
            f"사고발생시간은 {row['사고발생시간']}시 이며, 사고요일은 {row['사고발생요일']}요일 이고, 사고발생월은 {row['사고발생월']}월 입니다. "
        ),
        "answer": row["재발방지대책"] # 재발방지대책 및 향후조치계획
    },
    axis=1
)

combined_valid_data = valid.apply(
    lambda row: {
        "question": (
            f"공사종류 대분류 '{row['공사종류(대분류)']}', 중분류 '{row['공사종류(중분류)']}' 공사 중 "
            f"공종 대분류 '{row['공종(대분류)']}', 중분류 '{row['공종(중분류)']}' 작업에서 "
            f"사고객체 '{row['사고객체(대분류)']}'(중분류: '{row['사고객체(중분류)']}')와 관련된 사고가 발생했습니다. "
            f"작업 프로세스는 '{row['작업프로세스']}'이며, 사고 원인은 '{row['사고원인']}'이고 사고 원인 유형은 '{row['사고원인유형']}' 유형 입니다. "
            f"조사결과 본 사고의 인적사고유형은 '{row['인적사고']}'이며, 물적사고유형은 '{row['물적사고']}'입니다. "
            f"장소 대분류 '{row['장소(대분류)']}', 장소 중분류 '{row['장소(중분류)']}'에서 "
            f"부위 대분류 '{row['부위(대분류)']}', 부위 중분류 '{row['부위(중분류)']}' 사고가 발생하였습니다. "
            f"해당 사고 발생 당시 기온은 '{row['기온']}'이며, 발생일시는 '{row['발생일시']}'입니다. "
            f"사고발생시간은 {row['사고발생시간']}시 이며, 사고요일은 {row['사고발생요일']}요일 이고, 사고발생월은 {row['사고발생월']}월 입니다. "
            f"재발 방지 대책 및 향후 조치 계획은 무엇인가요?"
        ),
        "context": (
            f"공사종류 대분류 '{row['공사종류(대분류)']}', 중분류 '{row['공사종류(중분류)']}' 공사 중 "
            f"공종 대분류 '{row['공종(대분류)']}', 중분류 '{row['공종(중분류)']}' 작업에서 "
            f"사고객체 '{row['사고객체(대분류)']}'(중분류: '{row['사고객체(중분류)']}')와 관련된 사고가 발생했습니다. "
            f"작업 프로세스는 '{row['작업프로세스']}'이며, 사고 원인은 '{row['사고원인']}'이고 사고 원인 유형은 '{row['사고원인유형']}' 유형 입니다. "
            f"조사결과 본 사고의 인적사고유형은 '{row['인적사고']}'이며, 물적사고유형은 '{row['물적사고']}'입니다. "
            f"장소 대분류 '{row['장소(대분류)']}', 장소 중분류 '{row['장소(중분류)']}'에서 "
            f"부위 대분류 '{row['부위(대분류)']}', 부위 중분류 '{row['부위(중분류)']}' 사고가 발생하였습니다. "
            f"해당 사고 발생 당시 기온은 '{row['기온']}'이며, 발생일시는 '{row['발생일시']}'입니다. "
            f"사고발생시간은 {row['사고발생시간']}시 이며, 사고요일은 {row['사고발생요일']}요일 이고, 사고발생월은 {row['사고발생월']}월 입니다. "
        ),
        "answer": row["재발방지대책"] # 재발방지대책 및 향후조치계획
    },
    axis=1
)

combined_test_data = test.apply(
    lambda row: {
        "question": (
            f"공사종류 대분류 '{row['공사종류(대분류)']}', 중분류 '{row['공사종류(중분류)']}' 공사 중 "
            f"공종 대분류 '{row['공종(대분류)']}', 중분류 '{row['공종(중분류)']}' 작업에서 "
            f"사고객체 '{row['사고객체(대분류)']}'(중분류: '{row['사고객체(중분류)']}')와 관련된 사고가 발생했습니다. "
            f"작업 프로세스는 '{row['작업프로세스']}'이며, 사고 원인은 '{row['사고원인']}'이고 사고 원인 유형은 '{row['사고원인유형']}' 유형 입니다. "
            f"조사결과 본 사고의 인적사고유형은 '{row['인적사고']}'이며, 물적사고유형은 '{row['물적사고']}'입니다. "
            f"장소 대분류 '{row['장소(대분류)']}', 장소 중분류 '{row['장소(중분류)']}'에서 "
            f"부위 대분류 '{row['부위(대분류)']}', 부위 중분류 '{row['부위(중분류)']}' 사고가 발생하였습니다. "
            f"해당 사고 발생 당시 기온은 '{row['기온']}'이며, 발생일시는 '{row['발생일시']}'입니다. "
            f"사고발생시간은 {row['사고발생시간']}시 이며, 사고요일은 {row['사고발생요일']}요일 이고, 사고발생월은 {row['사고발생월']}월 입니다. "
            f"재발 방지 대책 및 향후 조치 계획은 무엇인가요?"
        ),
        "context": (
            f"공사종류 대분류 '{row['공사종류(대분류)']}', 중분류 '{row['공사종류(중분류)']}' 공사 중 "
            f"공종 대분류 '{row['공종(대분류)']}', 중분류 '{row['공종(중분류)']}' 작업에서 "
            f"사고객체 '{row['사고객체(대분류)']}'(중분류: '{row['사고객체(중분류)']}')와 관련된 사고가 발생했습니다. "
            f"작업 프로세스는 '{row['작업프로세스']}'이며, 사고 원인은 '{row['사고원인']}'이고 사고 원인 유형은 '{row['사고원인유형']}' 유형 입니다. "
            f"조사결과 본 사고의 인적사고유형은 '{row['인적사고']}'이며, 물적사고유형은 '{row['물적사고']}'입니다. "
            f"장소 대분류 '{row['장소(대분류)']}', 장소 중분류 '{row['장소(중분류)']}'에서 "
            f"부위 대분류 '{row['부위(대분류)']}', 부위 중분류 '{row['부위(중분류)']}' 사고가 발생하였습니다. "
            f"해당 사고 발생 당시 기온은 '{row['기온']}'이며, 발생일시는 '{row['발생일시']}'입니다. "
            f"사고발생시간은 {row['사고발생시간']}시 이며, 사고요일은 {row['사고발생요일']}요일 이고, 사고발생월은 {row['사고발생월']}월 입니다. "
        )
    },
    axis=1
)

# DataFrame으로 변환
combined_train_data = pd.DataFrame(list(combined_train_data))
combined_valid_data = pd.DataFrame(list(combined_valid_data))
combined_test_data = pd.DataFrame(list(combined_test_data))

In [11]:
def create_context(row, target_words):
    """
    사고 데이터에서 주요 정보를 포함하는 문장을 생성하는 함수.
    - 사전 전처리를 적용하여, 타겟 데이터(test)에서 존재하지 않는 값은 None으로 변경.
    - None, 'nan', '' 값이 포함된 문장은 제거하여 문장을 깔끔하게 유지.

    Parameters:
        row (pd.Series): 데이터셋의 한 행
        target_words (dict): 각 열별 허용된 단어 목록 (test 데이터 기반)

    Returns:
        str: 필터링된 문장
    """
    # 사전 전처리: `test` 기준으로 존재하는 값만 유지
    전처리대상후보변수 = [
        '공사종류', '인적사고', '물적사고', '공종', '사고객체', '작업프로세스', '장소', '부위', '사고원인',
        '공사종류(대분류)', '공사종류(중분류)', '공종(대분류)', '공종(중분류)', '사고객체(대분류)', '사고객체(중분류)',
        '장소(대분류)', '장소(중분류)', '부위(대분류)', '부위(중분류)'
    ]

    for col in 전처리대상후보변수:
        if row[col] not in target_words[col]:  # `test` 기준 값이 아니면 None 처리
            row[col] = None

    # 문장 생성
    sentences = [
        f"공사종류 대분류 '{row['공사종류(대분류)']}', 중분류 '{row['공사종류(중분류)']}' 공사 중",
        f"공종 대분류 '{row['공종(대분류)']}', 중분류 '{row['공종(중분류)']}' 작업에서",
        f"사고객체 '{row['사고객체(대분류)']}'(중분류: '{row['사고객체(중분류)']}')와 관련된 사고가 발생했습니다.",
        f"작업 프로세스는 '{row['작업프로세스']}'이며, 사고 원인은 '{row['사고원인']}'이고 사고 원인 유형은 '{row['사고원인유형']}' 유형입니다.",
        f"조사결과 본 사고의 인적사고유형은 '{row['인적사고']}'이며, 물적사고유형은 '{row['물적사고']}'입니다.",
        f"장소 대분류 '{row['장소(대분류)']}', 장소 중분류 '{row['장소(중분류)']}'에서",
        f"부위 대분류 '{row['부위(대분류)']}', 부위 중분류 '{row['부위(중분류)']}' 사고가 발생하였습니다.",
        f"해당 사고 발생 당시 기온은 '{row['기온']}'이며, 발생일시는 '{row['발생일시']}'입니다.",
        f"사고발생시간은 {row['사고발생시간']}시이며, 사고요일은 {row['사고발생요일']}요일이고, 사고발생월은 {row['사고발생월']}월입니다.",
        "재발 방지 대책 및 향후 조치 계획은 무엇인가요?"
    ]

    # None, 'nan', '' 값이 포함된 문장 필터링
    filtered_sentences = [s for s in sentences if not any(v in s for v in ["None", "nan", "''"])]

    # 문장 합치기
    return " ".join(filtered_sentences)


### `test` 기준 사전 전처리 목록 생성
target_words = {col: set(test[col].dropna().unique()) for col in [
    '공사종류', '인적사고', '물적사고', '공종', '사고객체', '작업프로세스', '장소', '부위', '사고원인',
    '공사종류(대분류)', '공사종류(중분류)', '공종(대분류)', '공종(중분류)', '사고객체(대분류)', '사고객체(중분류)',
    '장소(대분류)', '장소(중분류)', '부위(대분류)', '부위(중분류)'
]}

### 전처리 포함된 `create_context` 함수 적용
combined_train_data['context'] = train.apply(lambda row: create_context(row, target_words), axis=1)
combined_valid_data['context'] = valid.apply(lambda row: create_context(row, target_words), axis=1)
combined_test_data['context'] = test.apply(lambda row: create_context(row, target_words), axis=1)

In [12]:
from datasets import Dataset

"""
Dataset({
    features: ['gem_id', 'meaning_representation', 'target', 'references'],
    num_rows: 5103
})

- gem_id : index
- meaning_representation : answer
- target : question
- references : context

"""

train_dataset = Dataset.from_pandas(combined_train_data.reset_index())
eval_dataset = Dataset.from_pandas(combined_valid_data.reset_index())
test_dataset = Dataset.from_pandas(combined_test_data.reset_index())

#### 📂 LLM 선택

In [13]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from transformers import BitsAndBytesConfig

model_id = "MLP-KTLim/llama-3-Korean-Bllossom-8B"
model_max_length = 512

# 1. 8-bit 양자화 설정 (bnb_config)
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,  # 8-bit 양자화 적용
    bnb_8bit_compute_dtype=torch.float16,  # 연산은 FP16으로 진행
    bnb_8bit_use_double_quant=True,  # 더블 양자화 적용 (더 안정적)
)

# 2. 모델 로드 (8-bit 양자화 적용)
drive_path = "/content/drive/MyDrive/models2/"

if not os.path.exists(f"{drive_path}{model_id}"):
    print("모델 다운로드 중...")
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config)
    tokenizer = AutoTokenizer.from_pretrained(model_id, model_max_length=model_max_length, padding_side="left", add_eos_token=True)

    # 모델 저장
    model.save_pretrained(f"{drive_path}{model_id}")
    tokenizer.save_pretrained(f"{drive_path}{model_id}")
    print("모델 저장 완료!")
else:
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config)
    tokenizer = AutoTokenizer.from_pretrained(model_id, model_max_length=model_max_length, padding_side="left", add_eos_token=True)
    print("모델이 이미 저장되어 있습니다.")

# 3. Gradient Checkpointing 적용 (VRAM 절약)
model.gradient_checkpointing_enable()

모델 다운로드 중...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/710 [00:00<?, ?B/s]

`low_cpu_mem_usage` was None, now default to True since model is quantized.


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/172 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.1k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

모델 저장 완료!


####  📂 tokenizer 설정

In [14]:
%%time

# CPU times: user 1min 21s, sys: 14.6 s, total: 1min 36s
# Wall time: 1min 3s

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    model_max_length=model_max_length,
    padding_side="left",
    add_eos_token=True)
tokenizer.pad_token = tokenizer.eos_token

def tokenize(prompt):
    result = tokenizer(
        prompt,
        truncation=True,
        max_length=model_max_length,
        padding="max_length",
    )
    result["labels"] = result["input_ids"].copy()
    return result

def generate_and_tokenize_prompt(data_point):
    full_prompt =f"""
    ### 지침: 당신은 건설 안전 전문가입니다.
    - 건설현장에서 사고가 발생했습니다. 주어진 [사고정보]을 기반으로, 답변(재발 방지 대책 및 향후 조치 계획)을 작성하세요.

    ### 답변(재발 방지 대책 및 향후 조치 계획) 작성 양식
    - 베스트 [재발 방지 대책 및 향후 조치 계획] 예제를 참고하여 일관된 형식으로 답변하세요.
    - 존댓말을 절대 사용하지 마세요.
    - 질문에 대한 핵심 내용만 요약하여 간략하게 핵심만 작성하세요.
    - 프롬프트에 제공된 내용을 절대로 복사하지 마세요.
    - 답변은 최대 70자 내외로 작성하세요.

    ### 베스트 답변(재발 방지 대책 및 향후 조치 계획) 예시 3개:
    1. 작업 전 안전교육 실시와 안전점검 철저 지시를 통한 재발 방지 대책 마련.
    2. 이동통로 확보 관리와 작업 전 안전교육 철저 및 정기적 근로자 안전교육 시행을 통한 재발 방지 대책.
    3. 작업 시 안전교육 및 보호구 착용과 안전매트 설치를 통한 재발 방지 대책 및 향후 조치 계획.

    ### 사고정보:
    {data_point['context']}

    ### 질문:
    {data_point['question']}

    ### 답변(재발 방지 대책 및 향후 조치 계획):
    {data_point['answer']}
    """
    return tokenize(full_prompt)


tokenized_train_dataset = train_dataset.map(generate_and_tokenize_prompt)
tokenized_eval_dataset = eval_dataset.map(generate_and_tokenize_prompt)

# check
print("# question: " + train_dataset[1]['question'])
print("# answer: " + train_dataset[1]['answer'] + "\n")

Map:   0%|          | 0/14053 [00:00<?, ? examples/s]

Map:   0%|          | 0/9369 [00:00<?, ? examples/s]

# question: 공사종류 대분류 '건축', 중분류 '건축물' 공사 중 공종 대분류 '건축', 중분류 '수장공사' 작업에서 사고객체 '건설공구'(중분류: '사다리')와 관련된 사고가 발생했습니다. 작업 프로세스는 '설치작업'이며, 사고 원인은 '세대내에서 협소한 구간에서 환기배기관 설치를 위해 1M 사다리를 설치하고 종류 후 내려오던중 발을 헛디뎌 떨어져 우측 비골골절 및 요추통증으로 인해 06주 간의 입원경과 관찰이 발생된 재해'이고 사고 원인 유형은 '골절|발헛디딤' 유형 입니다. 조사결과 본 사고의 인적사고유형은 '떨어짐(2미터 미만)'이며, 물적사고유형은 '없음'입니다. 장소 대분류 '근린생활시설', 장소 중분류 '내부'에서 부위 대분류 '사다리', 부위 중분류 '상부(위)' 사고가 발생하였습니다. 해당 사고 발생 당시 기온은 '18℃'이며, 발생일시는 '2023-05-09 10:00:00'입니다. 사고발생시간은 10시 이며, 사고요일은 화요일 이고, 사고발생월은 5월 입니다. 재발 방지 대책 및 향후 조치 계획은 무엇인가요?
# answer: 사다리 작업 전 상태 및 안전조치 확인과 작업 시 2인 1조 철저 수행.

CPU times: user 56.5 s, sys: 836 ms, total: 57.3 s
Wall time: 56.9 s


#### 📂 프롬프트 설정 및 학습

In [15]:
eval_prompt = """
    ### 지침: 당신은 건설 안전 전문가입니다.
    - 건설현장에서 사고가 발생했습니다. 주어진 [사고정보]을 기반으로, 답변(재발 방지 대책 및 향후 조치 계획)을 작성하세요.

    ### 답변(재발 방지 대책 및 향후 조치 계획) 작성 양식
    - 베스트 [재발 방지 대책 및 향후 조치 계획] 예제를 참고하여 일관된 형식으로 답변하세요.
    - 존댓말을 절대 사용하지 마세요.
    - 질문에 대한 핵심 내용만 요약하여 간략하게 핵심만 작성하세요.
    - 프롬프트에 제공된 내용을 절대로 복사하지 마세요.
    - 답변은 최대 70자 내외로 작성하세요.

    ### 베스트 답변(재발 방지 대책 및 향후 조치 계획) 예시 3개:
    1. 작업 전 안전교육 실시와 안전점검 철저 지시를 통한 재발 방지 대책 마련.
    2. 이동통로 확보 관리와 작업 전 안전교육 철저 및 정기적 근로자 안전교육 시행을 통한 재발 방지 대책.
    3. 작업 시 안전교육 및 보호구 착용과 안전매트 설치를 통한 재발 방지 대책 및 향후 조치 계획.

    ### 사고정보:
    {data_point['context']}

    ### 질문:
    {data_point['question']}

    ### 답변(재발 방지 대책 및 향후 조치 계획):
    """

In [16]:
%%time

# CPU times: user 1min 21s, sys: 14.6 s, total: 1min 36s
# Wall time: 1min 3s

device = "cuda" # the device to load the model onto
model_input = tokenizer(eval_prompt, return_tensors="pt").to(device)
model.eval()
with torch.no_grad():
    print(tokenizer.decode(model.generate(**model_input, max_new_tokens=model_max_length, pad_token_id=2)[0], skip_special_tokens=True))


    ### 지침: 당신은 건설 안전 전문가입니다.
    - 건설현장에서 사고가 발생했습니다. 주어진 [사고정보]을 기반으로, 답변(재발 방지 대책 및 향후 조치 계획)을 작성하세요.

    ### 답변(재발 방지 대책 및 향후 조치 계획) 작성 양식 
    - 베스트 [재발 방지 대책 및 향후 조치 계획] 예제를 참고하여 일관된 형식으로 답변하세요.  
    - 존댓말을 절대 사용하지 마세요.
    - 질문에 대한 핵심 내용만 요약하여 간략하게 핵심만 작성하세요.
    - 프롬프트에 제공된 내용을 절대로 복사하지 마세요.   
    - 답변은 최대 70자 내외로 작성하세요. 

    ### 베스트 답변(재발 방지 대책 및 향후 조치 계획) 예시 3개:
    1. 작업 전 안전교육 실시와 안전점검 철저 지시를 통한 재발 방지 대책 마련.
    2. 이동통로 확보 관리와 작업 전 안전교육 철저 및 정기적 근로자 안전교육 시행을 통한 재발 방지 대책.
    3. 작업 시 안전교육 및 보호구 착용과 안전매트 설치를 통한 재발 방지 대책 및 향후 조치 계획.

    ### 사고정보:
    {data_point['context']}

    ### 질문:
    {data_point['question']}

    ### 답변(재발 방지 대책 및 향후 조치 계획):
     [여기에 답변 작성]  ```python
    import json

    def generate_answer(data_point):
        # 사고 정보 및 질문 내용 추출
        context = data_point['context']
        question = data_point['question']

        # 베스트 답변 예시 3개
        best_answers = [
            "작업 전 안전교육 실시와 안전점검 철저 지시를 통한 재발 방지 대책 마련.",
            "이동통로 확보 관리와 작업 전 안

In [17]:
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [18]:
# target_modules는 LoRA가 적용될 특정 모델의 레이어를 지정하는 옵션
# 모든 파라미터를 학습하는 것이 아니라, 특정 **레이어(모듈)**만 학습하도록 설정하여 VRAM을 절약하고 학습 속도를 높임

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",     # Query Projection (Self-Attention)
        "k_proj",     # Key Projection (Self-Attention)
        "v_proj",     # Value Projection (Self-Attention)
        "o_proj",     # Output Projection (Self-Attention)
        "gate_proj",  # Gating Mechanism in MLP
        "up_proj",    # MLP Up Projection (확장 레이어)
        "down_proj",  # MLP Down Projection (축소 레이어)
        "lm_head",    # Language Model Head (출력층)
    ],
    bias="none",
    lora_dropout=0.05,  # Conventional
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, config)
print_trainable_parameters(model)
# Apply the accelerator. You can comment this out to remove the accelerator.
# model = accelerator.prepare_model(model)

trainable params: 22030336 || all params: 8052291584 || trainable%: 0.27359088739129295


In [ ]:
%%time

# 4500 STEPS -> 7 hrs

from tqdm import tqdm
import transformers
from transformers import TrainerCallback

class ProgressCallback(TrainerCallback):
    """Custom callback to display training progress with tqdm progress bar."""

    def __init__(self, total_steps):
        self.progress_bar = tqdm(total=total_steps, desc="Training Progress", position=0, leave=True)

    def on_step_end(self, args, state, control, **kwargs):
        """Update progress bar at the end of each step."""
        self.progress_bar.update(1)  # 1 스텝 증가

    def on_train_end(self, args, state, control, **kwargs):
        """Close progress bar when training ends."""
        self.progress_bar.close()

# 프로젝트 설정
project = "dms-project7-1"
base_model_name = "llama-3-Korean-Bllossom-8B"
run_name = base_model_name + "-" + project
output_dir = drive_path + run_name
print('# drive_path:', drive_path)
print('# output_dir:', output_dir)

# 토크나이저 패딩 설정
tokenizer.pad_token = tokenizer.eos_token

# 학습 옵션
batch_size = 2
num_epochs = 1
num_train_steps = (len(tokenized_train_dataset) * 1) // 2  # 총 학습 스텝 계산
# max_steps = (len(tokenized_train_dataset) * num_epochs) // batch_size # max_steps
max_steps = 4500
print('# num_train_steps:',num_train_steps)
print('# max_steps:',max_steps)

trainer = transformers.Trainer(
    model=model,
    train_dataset = tokenized_train_dataset,
    eval_dataset = tokenized_eval_dataset,
    args=transformers.TrainingArguments(
        output_dir=output_dir,
        warmup_steps=5,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=4,

        # num_train_epochs=1,  # max_steps 대신 epochs 설정 (1회 반복)
        max_steps = max_steps, # 40%(4500/11599), 전체 23198 -> 학습 시간 결정 요인

        learning_rate=2.5e-5,
        logging_steps=500,
        bf16=True,
        optim="paged_adamw_8bit",
        logging_dir="./logs",
        save_strategy="steps",
        save_steps=500,
        evaluation_strategy="steps",
        eval_steps=500,
        do_eval=True,
        report_to='none',
        run_name=f"{run_name}-{datetime.now().strftime('%Y-%m-%d-%H-%M')}",
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
    callbacks=[ProgressCallback(total_steps=num_train_steps)],  # 프로그레스바 추가
)

# 파인튜닝 실행 (진행률 표시됨)
model.config.use_cache = False  # 캐시 경고 제거
trainer.train()

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


# drive_path: /content/drive/MyDrive/models2/
# output_dir: /content/drive/MyDrive/models2/llama-3-Korean-Bllossom-8B-dms-project7-1
# num_train_steps: 7026
# max_steps: 4500


Training Progress:   0%|          | 0/7026 [00:00<?, ?it/s]No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/bitsandbytes/autograd/_functions.py:315: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: 

Step,Training Loss,Validation Loss


Training Progress:   0%|          | 9/7026 [00:49<10:27:28,  5.37s/it]

#### 📂 파인튜닝모델 테스트

In [ ]:
%%time

# CPU times: user 48.8 s, sys: 880 ms, total: 49.7 s
# Wall time: 48.7 s

# output_dir: /content/drive/MyDrive/models/llama-3-Korean-Bllossom-8B-dms-project7

ft_model = PeftModel.from_pretrained(model, f"{output_dir}/checkpoint-4500")

ft_model.eval()
with torch.no_grad():
    print(tokenizer.decode(ft_model.generate(**model_input, max_new_tokens=256, pad_token_id=2)[0], skip_special_tokens=True,repetition_penalty=1.5,temperature=0.2))